# 03 · Feature Engineering

**Goal:** Extract four feature sets from preprocessed texts and save
them as ready-to-use matrices for classical ML models.

**Feature sets:**
1. **HC** — 68 hand-crafted stylometric features (POS tags, function words, punctuation, lexical complexity)
2. **TF-IDF word** — word-level n-gram TF-IDF
3. **TF-IDF char** — character-level n-gram TF-IDF
4. **hc_plus_word_char** — concatenation of HC + word TF-IDF + char TF-IDF

**Sections:**
1. Load preprocessed data
2. Hand-crafted features (HC)
3. TF-IDF vectorizers
4. Combine & save feature matrices
5. Feature statistics & distributions
6. Correlation analysis
7. Validation

In [ ]:

# ── 0. Imports & settings ────────────────────────────────────────────────

import json
import pathlib
import pickle
import time
import warnings
from collections import Counter
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
from IPython.display import display
from scipy import sparse
from scipy.stats import pointbiserialr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.bbox"] = "tight"

# ── Paths ──
def resolve_project_root() -> pathlib.Path:
    cwd = pathlib.Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for base in candidates:
        if (base / "data").exists() and (base / "outputs").exists():
            return base
    return cwd.parent

PROJECT_ROOT = resolve_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "figures" / "features"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"FEATURE_DIR   : {FEATURE_DIR}")


## 1 · Load preprocessed data

In [ ]:

splits = {}
for name in ["train", "val", "test"]:
    path = PROCESSED_DIR / f"{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run 02_text_preprocessing_pipeline.ipynb first.")
    splits[name] = pd.read_parquet(path)
    print(f"  {name}: {len(splits[name]):>6,} rows")

df_train = splits["train"].copy()
df_val = splits["val"].copy()
df_test = splits["test"].copy()

for split_name, df_split in [("train", df_train), ("val", df_val), ("test", df_test)]:
    assert "text" in df_split.columns, f"'text' column missing in {split_name}"
    assert "label" in df_split.columns, f"'label' column missing in {split_name}"
    assert df_split["text"].isnull().sum() == 0, f"Null texts found in {split_name}"

y_train = df_train["label"].astype(int).values
y_val = df_val["label"].astype(int).values
y_test = df_test["label"].astype(int).values

print(f"\nLabel balance (train): {np.bincount(y_train)} → {np.mean(y_train):.3f} positive rate")


## 2 · Hand-crafted features (HC)

68 stylometric features grouped into:
- **Lexical surface** (12): char count, word count, avg word length, sentence count, etc.
- **POS tag ratios** (17): NOUN, VERB, ADJ, ADV, PRON, DET, ADP, CONJ, NUM, etc.
- **Function word ratios** (20): top English function words
- **Punctuation ratios** (10): period, comma, exclamation, question, colon, semicolon, etc.
- **Lexical complexity** (9): TTR, corrected TTR, MTLD, Yule's K, hapax ratio, etc.

In [ ]:

# ── 2a. Load spaCy ──
MODEL_NAME = "en_core_web_sm"
try:
    nlp = spacy.load(MODEL_NAME, disable=["ner"])
except OSError:
    spacy.cli.download(MODEL_NAME)
    nlp = spacy.load(MODEL_NAME, disable=["ner"])

nlp.max_length = 500_000
print(f"spaCy model: {MODEL_NAME}")


In [ ]:

# ── 2b. Feature extraction functions ──

# Top 20 English function words
FUNCTION_WORDS = [
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "shall", "should",
]

# POS tags to track
POS_TAGS = [
    "NOUN", "VERB", "ADJ", "ADV", "PRON", "DET", "ADP", "CONJ", "CCONJ",
    "SCONJ", "NUM", "PART", "INTJ", "AUX", "PROPN", "PUNCT", "SYM",
]

# Punctuation characters to track
PUNCT_CHARS = [".", ",", "!", "?", ":", ";", "-", "'", '"', "("]

def _safe_div(a: float, b: float) -> float:
    return a / b if b != 0 else 0.0

def compute_ttr(tokens: List[str]) -> float:
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

def compute_corrected_ttr(tokens: List[str]) -> float:
    n = len(tokens)
    if n == 0:
        return 0.0
    return len(set(tokens)) / np.sqrt(2 * n)

def compute_hapax_ratio(tokens: List[str]) -> float:
    if not tokens:
        return 0.0
    freq = Counter(tokens)
    hapax = sum(1 for v in freq.values() if v == 1)
    return hapax / len(tokens)

def compute_yule_k(tokens: List[str]) -> float:
    if not tokens:
        return 0.0
    freq = Counter(tokens)
    n = len(tokens)
    spectrum = Counter(freq.values())
    m2 = sum(i * i * spectrum[i] for i in spectrum)
    if n <= 1:
        return 0.0
    return 10_000 * (m2 - n) / (n * n)

def compute_mtld(tokens: List[str], threshold: float = 0.72) -> float:
    if len(tokens) < 10:
        return 0.0

    def _mtld_one_direction(tok_list):
        factors = 0.0
        seen = set()
        segment_len = 0
        for t in tok_list:
            seen.add(t)
            segment_len += 1
            ttr_val = len(seen) / segment_len
            if ttr_val <= threshold:
                factors += 1.0
                seen = set()
                segment_len = 0
        if segment_len > 0:
            current_ttr = len(seen) / segment_len
            factors += (1.0 - current_ttr) / (1.0 - threshold) if threshold < 1.0 else 0.0
        return _safe_div(len(tok_list), factors)

    forward = _mtld_one_direction(tokens)
    backward = _mtld_one_direction(tokens[::-1])
    return (forward + backward) / 2.0

print("Feature functions defined.")


In [ ]:

# ── 2c. Main feature extractor ──

def extract_hc_features(doc) -> Dict[str, float]:
    """Extract 68 hand-crafted features from a spaCy Doc."""
    text = doc.text
    tokens_all = [tok for tok in doc]
    tokens_alpha = [tok.text.lower() for tok in doc if tok.is_alpha]
    sents = list(doc.sents)

    n_chars = len(text)
    n_tokens = len(tokens_all)
    n_alpha = len(tokens_alpha)
    n_sents = len(sents)

    feat = {}

    # ── Lexical surface (12) ──
    feat["n_chars"] = n_chars
    feat["n_tokens"] = n_tokens
    feat["n_alpha_tokens"] = n_alpha
    feat["n_sentences"] = n_sents
    feat["avg_word_len"] = _safe_div(sum(len(t) for t in tokens_alpha), n_alpha)
    feat["avg_sentence_len_words"] = _safe_div(n_tokens, n_sents)
    feat["avg_sentence_len_chars"] = _safe_div(n_chars, n_sents)

    word_lengths = [len(t) for t in tokens_alpha]
    feat["std_word_len"] = float(np.std(word_lengths)) if word_lengths else 0.0
    feat["max_word_len"] = max(word_lengths) if word_lengths else 0

    sent_lens = [len(list(s)) for s in sents]
    feat["std_sentence_len"] = float(np.std(sent_lens)) if sent_lens else 0.0

    feat["uppercase_ratio"] = _safe_div(sum(1 for c in text if c.isupper()), max(n_chars, 1))
    feat["digit_ratio"] = _safe_div(sum(1 for c in text if c.isdigit()), max(n_chars, 1))

    # ── POS tag ratios (17) ──
    pos_counts = Counter(tok.pos_ for tok in tokens_all)
    for pos in POS_TAGS:
        feat[f"pos_{pos.lower()}_ratio"] = _safe_div(pos_counts.get(pos, 0), n_tokens)

    # ── Function word ratios (20) ──
    lower_tokens = [t.text.lower() for t in tokens_all]
    token_freq = Counter(lower_tokens)
    for fw in FUNCTION_WORDS:
        feat[f"fw_{fw}"] = _safe_div(token_freq.get(fw, 0), n_tokens)

    # ── Punctuation ratios (10) ──
    char_freq = Counter(text)
    for pc in PUNCT_CHARS:
        safe_name = {
            ".": "period", ",": "comma", "!": "excl", "?": "question",
            ":": "colon", ";": "semicolon", "-": "dash", "'": "apos",
            '"': "dquote", "(": "paren",
        }.get(pc, pc)
        feat[f"punct_{safe_name}_ratio"] = _safe_div(char_freq.get(pc, 0), max(n_chars, 1))

    # ── Lexical complexity (9) ──
    feat["ttr"] = compute_ttr(tokens_alpha)
    feat["corrected_ttr"] = compute_corrected_ttr(tokens_alpha)
    feat["hapax_ratio"] = compute_hapax_ratio(tokens_alpha)
    feat["yule_k"] = compute_yule_k(tokens_alpha)
    feat["mtld"] = compute_mtld(tokens_alpha)

    lemmas = [tok.lemma_.lower() for tok in doc if tok.is_alpha]
    feat["avg_lemma_len"] = _safe_div(sum(len(l) for l in lemmas), len(lemmas))
    feat["lemma_ttr"] = compute_ttr(lemmas)

    n_stop = sum(1 for tok in doc if tok.is_stop)
    feat["stopword_ratio"] = _safe_div(n_stop, n_tokens)

    feat["short_word_ratio"] = _safe_div(sum(1 for t in tokens_alpha if len(t) <= 3), n_alpha)

    return feat

test_doc = nlp("Hello world! This is a test email from [URL]. Please click here.")
test_feats = extract_hc_features(test_doc)
print(f"Number of HC features: {len(test_feats)}")
print(f"Feature names: {list(test_feats.keys())[:10]}...")
assert len(test_feats) == 68, f"Expected 68 HC features, got {len(test_feats)}"


In [ ]:

# ── 2d. Extract HC features for all splits ──

def extract_hc_for_split(texts: List[str], split_name: str, batch_size: int = 256) -> pd.DataFrame:
    t0 = time.time()
    features = []
    for doc in nlp.pipe(texts, batch_size=batch_size, n_process=1):
        features.append(extract_hc_features(doc))
    elapsed = time.time() - t0
    speed = len(texts) / elapsed if elapsed > 0 else 0.0
    print(f"  {split_name}: {len(texts):,} texts → {elapsed:.1f}s ({speed:.0f} texts/s)")
    return pd.DataFrame(features)

print("Extracting HC features...")
hc_train = extract_hc_for_split(df_train["text"].tolist(), "train")
hc_val = extract_hc_for_split(df_val["text"].tolist(), "val")
hc_test = extract_hc_for_split(df_test["text"].tolist(), "test")

HC_FEATURE_NAMES = list(hc_train.columns)
print(f"\nTotal HC features: {len(HC_FEATURE_NAMES)}")
assert len(HC_FEATURE_NAMES) == 68, f"Expected 68 HC features, got {len(HC_FEATURE_NAMES)}"


In [ ]:

# ── 2e. Check for NaN / Inf ──
for name, df_hc in [("train", hc_train), ("val", hc_val), ("test", hc_test)]:
    n_nan = int(df_hc.isna().sum().sum())
    n_inf = int(np.isinf(df_hc.select_dtypes(include=[np.number]).values).sum())
    print(f"  {name}: NaN={n_nan}, Inf={n_inf}")

hc_train = hc_train.fillna(0.0)
hc_val = hc_val.fillna(0.0)
hc_test = hc_test.fillna(0.0)


In [ ]:

# ── 2f. Scale HC features ──
scaler = StandardScaler()
X_hc_train = scaler.fit_transform(hc_train.values)
X_hc_val = scaler.transform(hc_val.values)
X_hc_test = scaler.transform(hc_test.values)

print(f"HC shapes: train={X_hc_train.shape}, val={X_hc_val.shape}, test={X_hc_test.shape}")


## 3 · TF-IDF vectorizers

In [ ]:

# ── 3a. TF-IDF word n-grams ──
tfidf_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=30_000,
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents="unicode",
    dtype=np.float32,
)

t0 = time.time()
X_word_train = tfidf_word.fit_transform(df_train["text"])
X_word_val = tfidf_word.transform(df_val["text"])
X_word_test = tfidf_word.transform(df_test["text"])
elapsed = time.time() - t0

print(f"TF-IDF word: vocab={len(tfidf_word.vocabulary_):,}, time={elapsed:.1f}s")
print(f"  train: {X_word_train.shape}, val: {X_word_val.shape}, test: {X_word_test.shape}")


In [ ]:

# ── 3b. TF-IDF char n-grams ──
tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=50_000,
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32,
)

t0 = time.time()
X_char_train = tfidf_char.fit_transform(df_train["text"])
X_char_val = tfidf_char.transform(df_val["text"])
X_char_test = tfidf_char.transform(df_test["text"])
elapsed = time.time() - t0

print(f"TF-IDF char: vocab={len(tfidf_char.vocabulary_):,}, time={elapsed:.1f}s")
print(f"  train: {X_char_train.shape}, val: {X_char_val.shape}, test: {X_char_test.shape}")


In [ ]:

# ── 3c. Top TF-IDF terms ──
def top_tfidf_terms(vectorizer, matrix, label_array, top_n=15):
    feature_names = vectorizer.get_feature_names_out()
    for label_val, label_name in [(0, "Human"), (1, "LLM")]:
        mask = label_array == label_val
        mean_tfidf = np.asarray(matrix[mask].mean(axis=0)).flatten()
        top_idx = mean_tfidf.argsort()[-top_n:][::-1]
        terms = [(feature_names[i], mean_tfidf[i]) for i in top_idx]
        print(f"\n  {label_name} — top {top_n} terms:")
        for term, score in terms:
            print(f"    {term:30s} {score:.4f}")

print("=" * 50)
print("Top TF-IDF WORD terms (train):")
top_tfidf_terms(tfidf_word, X_word_train, y_train)

print("\n" + "=" * 50)
print("Top TF-IDF CHAR n-grams (train):")
top_tfidf_terms(tfidf_char, X_char_train, y_train)


## 4 · Combine & save feature matrices

In [ ]:

# ── 4a. Build combined feature sets ──

def build_combined(hc, word, char):
    hc_sparse = sparse.csr_matrix(hc)
    return sparse.hstack([hc_sparse, word, char], format="csr")

X_combined_train = build_combined(X_hc_train, X_word_train, X_char_train)
X_combined_val = build_combined(X_hc_val, X_word_val, X_char_val)
X_combined_test = build_combined(X_hc_test, X_word_test, X_char_test)

print("Combined (hc_plus_word_char) shapes:")
print(f"  train: {X_combined_train.shape}")
print(f"  val:   {X_combined_val.shape}")
print(f"  test:  {X_combined_test.shape}")
print(f"\nBreakdown: HC={X_hc_train.shape[1]} + word={X_word_train.shape[1]} + char={X_char_train.shape[1]} = {X_combined_train.shape[1]}")


In [ ]:

# ── 4b. Feature set registry ──
FEATURE_SETS = {
    "hc": {
        "train": sparse.csr_matrix(X_hc_train),
        "val": sparse.csr_matrix(X_hc_val),
        "test": sparse.csr_matrix(X_hc_test),
        "n_features": X_hc_train.shape[1],
        "feature_names": HC_FEATURE_NAMES,
    },
    "tfidf_word": {
        "train": X_word_train,
        "val": X_word_val,
        "test": X_word_test,
        "n_features": X_word_train.shape[1],
        "feature_names": list(tfidf_word.get_feature_names_out()),
    },
    "tfidf_char": {
        "train": X_char_train,
        "val": X_char_val,
        "test": X_char_test,
        "n_features": X_char_train.shape[1],
        "feature_names": list(tfidf_char.get_feature_names_out()),
    },
    "hc_plus_word_char": {
        "train": X_combined_train,
        "val": X_combined_val,
        "test": X_combined_test,
        "n_features": X_combined_train.shape[1],
        "feature_names": (
            HC_FEATURE_NAMES
            + [f"w_{n}" for n in tfidf_word.get_feature_names_out()]
            + [f"c_{n}" for n in tfidf_char.get_feature_names_out()]
        ),
    },
}

print("Feature set summary:")
for name, info in FEATURE_SETS.items():
    print(f"  {name:25s} → {info['n_features']:>6,} features")


In [ ]:

# ── 4c. Save feature matrices ──

for fs_name, fs_data in FEATURE_SETS.items():
    fs_dir = FEATURE_DIR / fs_name
    fs_dir.mkdir(parents=True, exist_ok=True)

    for split_name in ["train", "val", "test"]:
        mat = fs_data[split_name]
        out_path = fs_dir / f"X_{split_name}.npz"
        sparse.save_npz(out_path, mat if sparse.issparse(mat) else sparse.csr_matrix(mat))

    names_path = fs_dir / "feature_names.json"
    with open(names_path, "w", encoding="utf-8") as f:
        json.dump(fs_data["feature_names"], f, ensure_ascii=False)

    print(f"  Saved {fs_name}: {fs_dir}")

for split_name, y_arr in [("train", y_train), ("val", y_val), ("test", y_test)]:
    np.save(FEATURE_DIR / f"y_{split_name}.npy", y_arr)

print(f"\n  Labels saved: y_train, y_val, y_test")


In [ ]:

# ── 4d. Save fitted transformers for reproducibility ──
artifacts = {
    "tfidf_word": tfidf_word,
    "tfidf_char": tfidf_char,
    "hc_scaler": scaler,
}

artifacts_path = FEATURE_DIR / "fitted_transformers.pkl"
with open(artifacts_path, "wb") as f:
    pickle.dump(artifacts, f)
print(f"Fitted transformers saved → {artifacts_path}")


## 5 · Feature statistics & distributions

In [ ]:

# ── 5a. HC feature distributions: Human vs LLM ──
hc_all = pd.concat(
    [hc_train.assign(label=y_train),
     hc_val.assign(label=y_val),
     hc_test.assign(label=y_test)],
    ignore_index=True,
)
hc_all["label_str"] = hc_all["label"].map({0: "Human", 1: "LLM"})

human_means = hc_all[hc_all["label"] == 0][HC_FEATURE_NAMES].mean()
llm_means = hc_all[hc_all["label"] == 1][HC_FEATURE_NAMES].mean()
overall_std = hc_all[HC_FEATURE_NAMES].std().replace(0, 1)
effect_size = ((llm_means - human_means) / overall_std).abs().sort_values(ascending=False)

top_features = effect_size.head(16).index.tolist()
print("Top 16 HC features by effect size (|Cohen's d| approx):")
for feat_name in top_features:
    print(f"  {feat_name:35s}  d={effect_size[feat_name]:.3f}")


In [ ]:

# ── 5b. Box-plots of top 16 features ──
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes_flat = axes.flatten()

for i, feat_name in enumerate(top_features):
    ax = axes_flat[i]
    sns.boxplot(
        data=hc_all, x="label_str", y=feat_name,
        showfliers=False, ax=ax,
        palette={"Human": "#4C72B0", "LLM": "#DD8452"},
        width=0.5,
    )
    ax.set_title(feat_name, fontsize=9)
    ax.set_xlabel("")
    ax.set_ylabel("")

fig.suptitle("Top 16 discriminative HC features (by effect size)", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "top_hc_features_boxplot.png")
plt.show()


In [ ]:

# ── 5c. Feature sparsity ──
print("Feature matrix sparsity:")
for fs_name, fs_data in FEATURE_SETS.items():
    mat = fs_data["train"]
    if sparse.issparse(mat):
        nnz_ratio = mat.nnz / (mat.shape[0] * mat.shape[1])
    else:
        nnz_ratio = np.count_nonzero(mat) / mat.size
    print(f"  {fs_name:25s}: {nnz_ratio:.4f} ({nnz_ratio*100:.2f}% non-zero)")


## 6 · Correlation analysis (HC features)

In [ ]:

# ── 6a. Correlation matrix of HC features ──
corr_matrix = hc_all[HC_FEATURE_NAMES].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, cmap="RdBu_r", center=0,
    vmin=-1, vmax=1, ax=ax,
    xticklabels=True, yticklabels=True,
    linewidths=0.5, linecolor="white",
    cbar_kws={"shrink": 0.8},
)
ax.set_title("HC Feature Correlation Matrix", fontsize=13)
ax.tick_params(axis="both", labelsize=6)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "hc_correlation_matrix.png")
plt.show()


In [ ]:

# ── 6b. Highly correlated pairs ──
high_corr_threshold = 0.85
upper = corr_matrix.where(~mask)
high_pairs = []
for col in upper.columns:
    for idx in upper.index:
        val = upper.loc[idx, col]
        if pd.notna(val) and abs(val) >= high_corr_threshold and idx != col:
            high_pairs.append((idx, col, round(float(val), 3)))

high_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print(f"Highly correlated pairs (|r| ≥ {high_corr_threshold}): {len(high_pairs)}")
for f1, f2, r in high_pairs[:20]:
    print(f"  {f1:35s} × {f2:35s}  r={r:+.3f}")


In [ ]:

# ── 6c. Point-biserial correlation with label ──
pb_corrs = {}
for feat_name in HC_FEATURE_NAMES:
    r, p = pointbiserialr(hc_all["label"].values, hc_all[feat_name].values)
    pb_corrs[feat_name] = {"r": round(float(r), 4), "p": float(p)}

pb_df = pd.DataFrame(pb_corrs).T.sort_values("r", key=lambda s: s.abs(), ascending=False)
pb_df["abs_r"] = pb_df["r"].abs()

print("Top 20 features by point-biserial correlation with label:")
display(pb_df.head(20))

fig, ax = plt.subplots(figsize=(10, 6))
top20 = pb_df.head(20)
colors = ["#DD8452" if r > 0 else "#4C72B0" for r in top20["r"]]
ax.barh(range(len(top20)), top20["r"].values, color=colors)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index, fontsize=8)
ax.set_xlabel("Point-biserial correlation with label")
ax.set_title("Top 20 HC features correlated with label (+ → LLM, − → Human)")
ax.axvline(0, color="grey", linewidth=0.5)
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "hc_label_correlation.png")
plt.show()


## 7 · Validation

In [ ]:

# ── 7a. Verify saved files ──
print("Saved feature files:")
for p in sorted(FEATURE_DIR.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f"  {p.relative_to(FEATURE_DIR)}  ({size_mb:.1f} MB)")


In [ ]:

# ── 7b. Reload and verify shapes ──
for fs_name in FEATURE_SETS:
    for split_name in ["train", "val", "test"]:
        loaded = sparse.load_npz(FEATURE_DIR / fs_name / f"X_{split_name}.npz")
        expected = FEATURE_SETS[fs_name][split_name]
        assert loaded.shape == expected.shape, (
            f"Shape mismatch for {fs_name}/{split_name}: "
            f"{loaded.shape} vs {expected.shape}"
        )
    print(f"  ✅ {fs_name}: all splits match")

for split_name, y_expected in [("train", y_train), ("val", y_val), ("test", y_test)]:
    y_loaded = np.load(FEATURE_DIR / f"y_{split_name}.npy")
    assert np.array_equal(y_loaded, y_expected), f"Label mismatch for {split_name}"
print("  ✅ Labels: all splits match")


In [ ]:

# ── 7c. Save feature engineering metadata ──
fe_meta = {
    "feature_sets": {
        name: {
            "n_features": int(info["n_features"]),
            "train_shape": list(info["train"].shape),
            "val_shape": list(info["val"].shape),
            "test_shape": list(info["test"].shape),
        }
        for name, info in FEATURE_SETS.items()
    },
    "tfidf_word_params": {
        "analyzer": "word",
        "ngram_range": [1, 2],
        "max_features": 30_000,
        "min_df": 3,
        "max_df": 0.95,
        "sublinear_tf": True,
        "actual_vocab_size": int(len(tfidf_word.vocabulary_)),
    },
    "tfidf_char_params": {
        "analyzer": "char_wb",
        "ngram_range": [3, 5],
        "max_features": 50_000,
        "min_df": 3,
        "max_df": 0.95,
        "sublinear_tf": True,
        "actual_vocab_size": int(len(tfidf_char.vocabulary_)),
    },
    "hc_scaler": "StandardScaler",
    "spacy_model": MODEL_NAME,
    "n_hc_features": int(len(HC_FEATURE_NAMES)),
    "hc_feature_names": HC_FEATURE_NAMES,
    "high_corr_pairs_count": int(len(high_pairs)),
}

meta_path = FEATURE_DIR / "feature_engineering_meta.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(fe_meta, f, indent=2, ensure_ascii=False)
print(f"Metadata saved → {meta_path}")


In [ ]:

# ── 7d. Summary table for thesis ──
summary_rows = []
for name, info in FEATURE_SETS.items():
    summary_rows.append({
        "Feature set": name,
        "# Features": info["n_features"],
        "Train samples": info["train"].shape[0],
        "Val samples": info["val"].shape[0],
        "Test samples": info["test"].shape[0],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TABLE_DIR / "feature_set_summary.csv", index=False)
print(f"Summary table saved → {TABLE_DIR / 'feature_set_summary.csv'}")
display(summary_df)


In [ ]:

print("\n" + "=" * 60)
print("Feature engineering notebook complete.")
print(f"Feature matrices : {FEATURE_DIR}")
print(f"Figures          : {OUTPUT_DIR}")
print(f"Tables           : {TABLE_DIR}")
print("=" * 60)
